# Initialization

In [9]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

In [4]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
os.environ["USER_AGENT"] = "retrival_agent"

In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Tools

In [ ]:
#Data importation 
doc_path = "./datasets/tunisia.txt"
loader = TextLoader(doc_path, encoding="utf-8")
documents = loader.load()

#First tool
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=1)
texts = text_splitter.split_documents(documents)

Created a chunk of size 1347, which is longer than the specified 1000
Created a chunk of size 2031, which is longer than the specified 1000
Created a chunk of size 1479, which is longer than the specified 1000
Created a chunk of size 2087, which is longer than the specified 1000
Created a chunk of size 1100, which is longer than the specified 1000
Created a chunk of size 1186, which is longer than the specified 1000
Created a chunk of size 1138, which is longer than the specified 1000
Created a chunk of size 1101, which is longer than the specified 1000
Created a chunk of size 1527, which is longer than the specified 1000
Created a chunk of size 1294, which is longer than the specified 1000
Created a chunk of size 1195, which is longer than the specified 1000
Created a chunk of size 1174, which is longer than the specified 1000
Created a chunk of size 1150, which is longer than the specified 1000
Created a chunk of size 1232, which is longer than the specified 1000
Created a chunk of s

In [65]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings

In [66]:
#Data embedding
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)
docsearch = Chroma.from_documents(texts, embeddings, collection_name="tunisian_history")

In [67]:
from langchain.chains import RetrievalQA

In [68]:
tunisian_history = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [69]:
#Second tool 
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://en.wikipedia.org/wiki/Wikipedia:FAQ/Readers")

In [70]:
docs = loader.load()
wiki_texts = text_splitter.split_documents(docs)
wiki_db = Chroma.from_documents(wiki_texts, embeddings, collection_name="wiki")

wiki = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=wiki_db.as_retriever()
)

Created a chunk of size 1832, which is longer than the specified 1000
Created a chunk of size 1292, which is longer than the specified 1000
Created a chunk of size 3365, which is longer than the specified 1000
Created a chunk of size 1833, which is longer than the specified 1000


In [71]:
#Third tool 
doc_path =  "./datasets/state_of_the_union.txt"

loader = TextLoader(doc_path, encoding="utf-8")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

docsearch = Chroma.from_documents(texts, embeddings, collection_name="state-of-union")

state_of_union = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

# Agent creation

In [72]:
from langchain.agents import AgentType, Tool, initialize_agent
from langchain_openai import OpenAI

In [ ]:
tools = [
    Tool(
        name="Tunisia history",
        func=tunisian_history.run,
        description="useful for when you need to answer questions about Tunisia history. Input should be a fully formed question.",
    ),
    Tool(
        name="Wikipedia QA System",
        func=wiki.run,
        description="useful for when you need to answer questions about Wikipedia. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
            name="State of Union QA System",
            func=state_of_union.run,
            description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
        ),
]

In [74]:
#Inizialise agent 
agent = initialize_agent(
    tools, llm, 
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, 
    handle_parsing_errors=True, 
    verbose=True
)

In [75]:
agent.invoke(
    "Who was the first Tunisian president?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to find information about the first president of Tunisia. This is a historical question, so I will use the Tunisia history tool to get accurate information.  
Action: Tunisia history  
Action Input: Who was the first Tunisian president?  
Observation: Tunisia history is not a valid tool, try one of [Tunisa history, Wikipedia QA System, State of Union QA System].
Thought:I need to use the correct spelling for the Tunisia history tool. I will try again with the correct spelling.  
Action: Tunisa history  
Action Input: Who was the first Tunisian president?  
Observation: The first Tunisian president was Habib Bourguiba.
Thought:I now know the final answer.  
Final Answer: The first Tunisian president was Habib Bourguiba.

> Finished chain.


{'input': 'Who was the first Tunisian president?',
 'output': 'The first Tunisian president was Habib Bourguiba.'}

In [76]:
agent.invoke(
    "Who is legit of German visa?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to gather information about the legitimacy of German visas, which may involve checking official sources or general knowledge about visa regulations. This seems to be a topic that might be covered in a general knowledge database like Wikipedia. 

Action: Wikipedia QA System  
Action Input: Who is eligible for a German visa?  
Observation: I don't know.


> Finished chain.


{'input': 'Who is legit of German visa?', 'output': "I don't know."}

In [77]:
agent.invoke(
    "Is Wikipedia censored?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


To answer the question about whether Wikipedia is censored, I should look for information regarding Wikipedia's editorial policies, controversies, and any instances of censorship that have been reported. This seems to fall under general knowledge, so I will use the Wikipedia QA System to find a comprehensive answer.

Action: Wikipedia QA System  
Action Input: Is Wikipedia censored?  
Observation: The policy of Wikipedia is to not censor content. The community of Wikipedia editors determines what text and images are displayed, so there may be material that offends some people. However, online access to Wikipedia itself may be censored by the governments of various countries. In such cases, using a VPN might be required to access Wikipedia.


> Finished chain.


{'input': 'Is Wikipedia censored?',
 'output': 'The policy of Wikipedia is to not censor content. The community of Wikipedia editors determines what text and images are displayed, so there may be material that offends some people. However, online access to Wikipedia itself may be censored by the governments of various countries. In such cases, using a VPN might be required to access Wikipedia.'}

In [78]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to find out what President Biden said about Ketanji Brown Jackson in the most recent State of the Union address. 
Action: State of Union QA System
Action Input: What did Biden say about Ketanji Brown Jackson in the State of the Union address?
Observation: In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson, describing her as one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence.
Thought:I now know the final answer.  
Final Answer: In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson, describing her as one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence.

> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': "In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson, describing her as one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence."}